# Applications of DigiMicPy to Specific Systems 


The general DigiMicPy models can be applied to specific microbial systems, ranging from animal gut microbiomes to aquatic microbial communities. To apply the software to specific hosts or environments, typical modifications may include: 

- Environmental parameters (e.g., temperature range)
- Microbial community structure (e.g., uptake matrix)
- Downstream analysis fit for the specific system (e.g., local stability)

Specifically, temperature-dependence and spatial distribtion are important features of these applications. 

# Example 1: Mice Gut Microbiome 


The mice gut microbiome is mainly composed of Bacteroidetes and Firmicutes. Being a complex real-world system, it is extremely challenging to track the dynamics of each species using the MiCRM or GLV. However, the general microbial models can still be modified and applied to model mice microbial communities, and simulate their responses to perturbations like infection-induced temperature changes. 

Below shows one example of applying general microbial community models to the mice gut microbiome, with a focus on temperature-dependent changes and spatial distribution of gut compartments. 

In [ ]:
# import packages 

import numpy as np
from scipy.stats import multivariate_normal
from scipy.integrate import solve_ivp

In [ ]:
# temperature-dependent uptake and respiration rates: parameters and functions 

def randtemp_param(N, kw): # generate random temperature-dependent traits for consumer species 
    rng=kw.get('rng',np.random) 

    L = kw['L'] # leakage 
    rho_t = kw['rho_t'] # correlation coefficient for covariance between activation energy and baseline uptake / mortality rates 
    L_v = np.mean(L)
    B0_m = -1.4954 # baseline mortality / respiration rate 
    B0_CUE = 0.1953 # baseline carbon use efficiency parameter 
    B0_u = np.log(np.exp(B0_m) / (1 - L_v - B0_CUE)) # baseline uptake rate 
    B0 = np.array([B0_u, B0_m]) 
    B0_var = 0.17 * np.abs(B0) 
    E_mean = np.array([0.8146, 0.5741]) # mean activation energy for uptake and respiration 
    E_var = 0.1364 * E_mean 
    cov_xy = rho_t * np.sqrt(B0_var * E_var) # covariance between activation energy and baseline uptake / mortality rates

    cov_u = np.array([[B0_var[0], cov_xy[0]], [cov_xy[0], E_var[0]]]) # covariance matrix for uptake
    cov_m = np.array([[B0_var[1], cov_xy[1]], [cov_xy[1], E_var[1]]]) # covariance matrix for respiration

    allu = multivariate_normal.rvs(mean=[B0[0], E_mean[0]], cov=cov_u, size=N).T # draw random samples from multivariate normal distribution for uptake
    allm = multivariate_normal.rvs(mean=[B0[1], E_mean[1]], cov=cov_m, size=N).T # draw random samples from multivariate normal distribution for respiration

    B = np.column_stack((np.exp(allu[0]), np.exp(allm[0]))) # exponentiate the base rates to get the actual values
    E = np.column_stack((allu[1], allm[1])) # activation energy 

    Tpu = 273.15 + rng.normal(35, 5, N) # draw random peak temperatures for uptake from a normal distribution with mean 35 and std 5
    Tpm = Tpu + 3 # peak temperature for respiration is 3 degrees higher than for uptake
    Tp = np.column_stack((Tpu, Tpm)) 

    return B, E, Tp


def temp_trait(N, kw):
    
    k = 0.0000862 
    T = kw['T']
    Tr = kw['Tr']
    Ed = kw['Ed']

    B, E, Tp = randtemp_param(N, kw) 
    
    # Arrhenius function with high-temp deactivation

    # uptake rate u(T)
    temp_p_u = B[:, 0] * np.exp((-E[:, 0] / k) * ((1 / T) - (1 / Tr))) / \
              (1 + (E[:, 0] / (Ed - E[:, 0])) * np.exp(Ed / k * (1 / Tp[:, 0] - 1 / T)))

    # respiration rate m(T)
    temp_p_m = B[:, 1] * np.exp((-E[:, 1] / k) * ((1 / T) - (1 / Tr))) / \
              (1 + (E[:, 1] / (Ed - E[:, 1])) * np.exp(Ed / k * (1 / Tp[:, 1] - 1 / T)))

    temp_p = np.column_stack((temp_p_u, temp_p_m))  

    return temp_p, B, E, Tp


# MiCRM default functions before parameter generation 


def def_m(N, M, kw):
    return np.ones(N)


def def_rho(N, M, kw):
    return np.ones(M)


def def_omega(N, M, kw):
    return np.ones(M)


def def_u(N, M, kw): # use if no modularity 
    rng = kw.get('rng', np.random)
    return rng.dirichlet(np.ones(M), size=N)


def def_l(N, M, kw): # use if no modularity 
    L = kw['L']
    rng = kw.get('rng', np.random) 
    l = np.zeros((N, M, M))
    phi = np.ones(M)
    for i in range(N):
        for alpha in range(M):
            draw = rng.dirichlet(alpha=phi)
            l[i, alpha, :] = draw * L[i]
    return l


In [ ]:
# modular uptake and leakage 

def modular_uptake(N, M, N_modules, s_ratio):
    assert N_modules <= M and N_modules <= N, "N_modules must be less than or equal to both M and N"

    # Baseline calculations
    sR = M // N_modules
    dR = M - (N_modules * sR)

    sC = N // N_modules
    dC = N - (N_modules * sC)

    # Get module sizes for M
    diffR = np.full(N_modules, sR, dtype=int)
    diffR[np.random.choice(N_modules, dR, replace=False)] += 1
    mR = [list(range(x - 1, y)) for x, y in zip((np.cumsum(diffR) - diffR + 1), np.cumsum(diffR))]

    # Get module sizes for N
    diffC = np.full(N_modules, sC, dtype=int)
    diffC[np.random.choice(N_modules, dC, replace=False)] += 1
    mC = [list(range(x - 1, y)) for x, y in zip((np.cumsum(diffC) - diffC + 1), np.cumsum(diffC))]

    # Preallocate u matrix
    u = np.random.rand(N, M)

    # Apply scaling
    for x, y in zip(mC, mR):
        u[np.ix_(x, y)] *= s_ratio

    # Normalize each row
    for i in range(N):
        u[i, :] /= np.sum(u[i, :])

    return u


def def_u_modular(N, M, kw):
    return modular_uptake(
        N,
        M,
        kw["N_modules"],
        kw["s_ratio"]
    )

def modular_leakage(M, N_modules, s_ratio, λ):
    assert N_modules <= M, "N_modules must be less than or equal to M"

    # Baseline
    sR = M // N_modules
    dR = M - (N_modules * sR)

    # Get module sizes and add to make to M
    diffR = np.full(N_modules, sR, dtype=int)
    diffR[np.random.choice(N_modules, dR, replace=False)] += 1
    mR = [list(range(x - 1, y)) for x, y in zip((np.cumsum(diffR) - diffR + 1), np.cumsum(diffR))]

    l = np.random.rand(M, M)

    for i, x in enumerate(mR):
        for j, y in enumerate(mR):
            if i == j or i + 1 == j:
                l[np.ix_(x, y)] *= s_ratio

    for i in range(M):
        l[i, :] = λ * l[i, :] / np.sum(l[i, :])

    return l


def generate_l_tensor(N, M, N_modules, s_ratio, λ):
    l_tensor = np.array([modular_leakage(M, N_modules, s_ratio, λ) for _ in range(N)])
    return l_tensor


def def_l_modular(N, M, kw):
    return generate_l_tensor(
        N,
        M,
        kw["N_modules"],
        kw["s_ratio"],
        np.mean(kw["L"])
    )


In [ ]:
# generate parameters

def generate_params(N,
                     M,
                     f_m=def_m,
                     f_rho=def_rho,
                     f_omega=def_omega,
                     f_u=def_u_modular,
                     f_l=def_l_modular,
                     **kwargs):


    kw = dict(kwargs)
    tt, B, E, Tp = temp_trait(N, kw) 
    kw['tt'] = tt

 
    m = f_m(N, M, kw) 
    u = f_u(N, M, kw) 
    l = f_l(N, M, kw)     


    lambda_ = np.sum(l, axis=2) 

 
    rho = f_rho(N, M, kw)
    omega = f_omega(N, M, kw)


    params = {
        'N': N,
        'M': M,
        'u': u,
        'm': m,
        'l': l,
        'rho': rho,
        'omega': omega,
        'lambda': lambda_,
        'L': kw['L'],
        'B': B,
        'E': E,
        'Tp': Tp,
        'tt': tt
    }
   
    params.update(kwargs)

    return params 
